# AIME — SD 3.5 Generation, local full fp16 (RTX 4090 24GB)

Runs SD 3.5 Large / Medium / Turbo on the 81 image-compatible AIME prompts, **full fp16, no quantization, no offload** — a 24GB card fits the whole pipeline (8B transformer ~16GB + T5-XXL text encoder ~10GB + CLIP + VAE) directly on-GPU.

**Different from `generate_SD35_colab.ipynb`**: that one runs on a 15GB Colab T4 and needs 4-bit NF4 quantization + dropping the T5-XXL text encoder to fit. Here, with 24GB, neither trick is needed — this keeps T5-XXL (better prompt fidelity) and full-precision weights (better image quality). The two are not pixel-identical; the existing 207 images already generated on Colab were quantized. Minor inconsistency, flagged for the record — not worth regenerating those 207 just for this.

**Bug fixed vs the original run**: the resume/dedup logic used to key only on `prompt_id`, ignoring `prompt_type`. Since Ambiguous and Explicit prompts reuse the same numeric IDs (both go 1..80ish), this silently skipped 12 real Explicit prompts whose ID matched an already-done Ambiguous prompt (found 2026-07-14, same bug independently found and fixed in `generate_gemini_flash.ipynb`). Fixed here from the start — the dedup key is now `(prompt_type, prompt_id, img_index)`.

## Before running
1. Copy the whole `AIME-public` repo (or at least `prompts/all_prompts.xlsx` and this `generation/` folder) to this PC.
2. Copy the existing generation progress so this run resumes instead of starting over: from `C:\Users\giand\Desktop\Annotazione\AIME\dataset\`, copy `labels/generated_SD35_Large.csv`, `labels/generated_SD35_Medium.csv`, `labels/generated_SD35_Turbo.csv` and the `media/images/SD35_Large/`, `SD35_Medium/`, `SD35_Turbo/` folders (207 images total) into this repo's `AIME-public/dataset/` in the same relative structure.
3. Put your HuggingFace token in `AIME-public/.env` as `HF_TOKEN=hf_...` (needs read access to the gated `stabilityai/stable-diffusion-3.5-*` repos — accept the licenses on huggingface.co first if not already done).
4. `pip install -U diffusers transformers accelerate huggingface_hub pandas openpyxl tqdm python-dotenv` (no `bitsandbytes` needed here — nothing is quantized).
5. Set `MODEL_KEY` below to `'large'`, `'medium'`, or `'turbo'` and run — it will print "Resuming: 69 images already generated" and only generate the 12 missing prompts if step 2 was done correctly. Repeat for the other two model keys.

In [ ]:
# ── Configuration (edit these) ───────────────────────────────────────
import os
from dotenv import load_dotenv
load_dotenv()

# Choose one: 'large' | 'medium' | 'turbo' -- run all three, one at a time
MODEL_KEY = 'turbo'

MODEL_MAP = {
    'large'  : ('stabilityai/stable-diffusion-3.5-large',       'SD35_Large'),
    'medium' : ('stabilityai/stable-diffusion-3.5-medium',      'SD35_Medium'),
    'turbo'  : ('stabilityai/stable-diffusion-3.5-large-turbo', 'SD35_Turbo'),
}
HF_MODEL_ID, MODEL_LABEL = MODEL_MAP[MODEL_KEY]
HF_TOKEN = os.environ.get('HF_TOKEN')

IMAGES_PER_PROMPT   = 1
NUM_INFERENCE_STEPS = 28   # 28 for large/medium; 4 for turbo (forced below)
GUIDANCE_SCALE      = 7.5  # ignored by turbo (CFG=0)

# Repo root -- adjust if this notebook isn't inside AIME-public/generation/ on this PC
REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROMPTS_XLS = os.path.join(REPO_ROOT, 'prompts', 'all_prompts.xlsx')
OUTPUT_DIR  = os.path.join(REPO_ROOT, 'dataset', 'media', 'images', MODEL_LABEL)
INDEX_CSV   = os.path.join(REPO_ROOT, 'dataset', 'labels', f'generated_{MODEL_LABEL}.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(INDEX_CSV), exist_ok=True)

print(f'Model   : {HF_MODEL_ID}')
print(f'Label   : {MODEL_LABEL}')
print(f'Output  : {OUTPUT_DIR}')
print(f'Index   : {INDEX_CSV}')

In [ ]:
# ── GPU check ──────────────────────────────────────────────────────
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True, timeout=10)
    print(out.stdout.strip())
except Exception as e:
    print(f'Could not query nvidia-smi ({e}); check GPU availability manually.')

In [ ]:
# ── Load prompts ────────────────────────────────────────────────
import pandas as pd

df_all = pd.read_excel(PROMPTS_XLS, sheet_name='All Prompts (100)')
df_all.columns = df_all.columns.str.strip()

df_img = df_all[df_all['Modality'].isin(['Image', 'Both'])].copy().reset_index(drop=True)

print(f'Total prompts    : {len(df_all)}')
print(f'Image-compatible : {len(df_img)}')
print(df_img.groupby(['Type','User Profile']).size().to_string())

In [ ]:
# ── Load model (full fp16, no quantization, no offload -- 24GB is plenty) ───────
import torch
from diffusers import StableDiffusion3Pipeline
from huggingface_hub import login

if HF_TOKEN:
    login(HF_TOKEN)

pipe = StableDiffusion3Pipeline.from_pretrained(
    HF_MODEL_ID,
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
pipe = pipe.to('cuda')
pipe.set_progress_bar_config(disable=True)
print(f'{MODEL_LABEL} loaded on {pipe.device}')

In [ ]:
# ── Generation helpers ────────────────────────────────────────────
import hashlib, datetime

def make_filename(prompt_id, prompt_type, index=0):
    ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S%f')
    h  = hashlib.sha256(f'{prompt_id}_{index}_{ts}'.encode()).hexdigest()[:10]
    t  = 'amb' if prompt_type == 'Ambiguous' else 'exp'
    return f'P{prompt_id:03d}_{t}_{h}.png'

print('Helpers ready.')

In [ ]:
# ── Resume: load already-generated index ────────────────────────────
# NOTE: dedup key includes prompt_type, not just prompt_id -- numeric IDs repeat across
# Ambiguous/Explicit, and keying on prompt_id alone silently skips real prompts. This is
# exactly the bug that caused 12 Explicit prompts to never be attempted on Colab -- fixed
# here from the start.
if os.path.exists(INDEX_CSV):
    df_done  = pd.read_csv(INDEX_CSV)
    done_ids = set(zip(df_done['prompt_type'], df_done['prompt_id'], df_done['img_index']))
    print(f'Resuming: {len(df_done)} images already generated.')
else:
    df_done  = pd.DataFrame()
    done_ids = set()
    print('Starting fresh.')

In [ ]:
# ── Main generation loop ──────────────────────────────────────────
from tqdm import tqdm

_guidance = 0.0 if MODEL_KEY == 'turbo' else GUIDANCE_SCALE
_steps    = 4   if MODEL_KEY == 'turbo' else NUM_INFERENCE_STEPS

records = []

for _, row in tqdm(df_img.iterrows(), total=len(df_img), desc=MODEL_LABEL):
    pid         = int(row['ID'])
    ptype       = row['Type']
    prompt_text = str(row['Prompt Text'])

    for img_idx in range(IMAGES_PER_PROMPT):
        if (ptype, pid, img_idx) in done_ids:
            continue

        status = 'ok'
        fname  = ''
        try:
            with torch.no_grad():
                result = pipe(
                    prompt                = prompt_text,
                    num_inference_steps   = _steps,
                    guidance_scale        = _guidance,
                    num_images_per_prompt = 1,
                )
            image = result.images[0]
            fname = make_filename(pid, ptype, img_idx)
            image.save(os.path.join(OUTPUT_DIR, fname))
            done_ids.add((ptype, pid, img_idx))
            torch.cuda.empty_cache()
        except Exception as e:
            print(f'  Error P{pid:03d}: {str(e)[:200]}')
            status = 'error'

        records.append({
            'file_path'   : f'images/{MODEL_LABEL}/{fname}' if fname else '',
            'prompt_id'   : pid,
            'img_index'   : img_idx,
            'prompt_type' : ptype,
            'user_profile': row['User Profile'],
            'modality'    : row['Modality'],
            'category'    : row['Category'],
            'language'    : row['Language'],
            'status'      : status,
            'model'       : MODEL_LABEL,
            'prompt_text' : prompt_text,
        })

        # Persist after every image
        df_new = pd.DataFrame(records)
        df_out = pd.concat([df_done, df_new], ignore_index=True) if not df_done.empty else df_new
        df_out.to_csv(INDEX_CSV, index=False)

ok  = (df_out['status'] == 'ok').sum()
err = (df_out['status'] == 'error').sum()
print(f'\nDone. Generated: {ok} | Errors: {err}')
print(f'Index: {INDEX_CSV}')

In [ ]:
# ── Summary ──────────────────────────────────────────────────
df_results = pd.read_csv(INDEX_CSV)
print(f'=== {MODEL_LABEL} Generation Summary ===')
print(df_results.groupby(['prompt_type','status']).size().unstack(fill_value=0).to_string())
print(f"\nTotal images saved: {(df_results['status']=='ok').sum()} / {len(df_results)}")